# MC-Dropout U-Net — uncertainty (Task04 Hippocampus)

Same skeleton as `unet-run.ipynb`, but for the plain MC-Dropout U-Net: minimal 2-channel preprocessing, train (early stopping), classic metrics, and Monte-Carlo-Dropout uncertainty (per-class + foreground calibration, entropy / mutual-information / variance maps).

**Before running:** make sure the new files (`train_mc.py`, `test_mc.py`, `uncertainty_mc.py`, `run_preprocessing_mc.py`, `mc_common.py`, `networks/UNET_mc.py`, `utilities/mc_dropout.py`, `datasets/preprocessing_plain.py`) are **committed & pushed** to the GitHub repo cloned below — the notebook runs them from the clone, not from your local disk.

Run the cells top to bottom, once each.

In [ ]:
# ===== init =====
# Path to the RAW MSD Task04 dataset (imagesTr/ labelsTr/ dataset.json) added as a Kaggle input dataset.
DATA_SRC = "/kaggle/input/datasets/thanasisrigas/task04-hippocampus/Task04_Hippocampus"  # <-- adjust to your input path

import os
os.environ["TASK"] = "Task04_Hippocampus"   # so config resolves the right task dir + labels

# MC-Dropout hyper-parameters (MUST be identical across train / test / uncertainty)
FOLDS      = [0]        # start with one fold; use [0, 1, 2, 3, 4] for full 5-fold CV
DROPOUT_P  = 0.4        # dropout prob at the 5 deep dropout points (encoder skips + bottleneck + decoder)
MC_SAMPLES = 30         # number of stochastic passes T at uncertainty time
OUT_DIR    = "results"  # weights / scores / uncertainty (inside the repo -> also fold-mean works)

In [ ]:
# ===== repo + requirements =====
# Clones the repo into ./repo and installs deps. (Internet must be ON in the notebook settings.)
!rm -rf repo
!git clone "https://github.com/ThanoSnake/my_unet.git" repo
%cd repo
!pip install -q medpy batchgenerators==0.21

In [ ]:
# ===== setup =====
# Copy the (read-only) raw dataset into config.DATA_DIR so preprocessing can write there,
# and quick-check that the MC model builds and the labels were read correctly.
import torch, os, shutil
import config
from networks.UNET_mc import MCDropoutUNet

# smoke test: 1-channel input -> num_classes logits
_ = MCDropoutUNet(num_classes=config.NUM_CLASSES, in_channels=1, dropout_p=DROPOUT_P)(torch.rand(2, 1, 64, 64))

dst = config.DATA_DIR
os.makedirs(dst.parent, exist_ok=True)
if not dst.exists():
    shutil.copytree(DATA_SRC, dst)
print("DATA_DIR:", dst)
print("num_classes:", config.NUM_CLASSES, "| labels:", config.LABELS)

In [ ]:
# ===== preprocessing (minimal, 2-channel image+label) =====
# Writes preprocessed/*.npy + splits.pkl into config.DATA_DIR. Run ONCE.
# (No morphology -> half the disk, no wasted compute vs the morph pipeline.)
!python run_preprocessing_mc.py

In [ ]:
# ===== train =====
# Trains one MC-Dropout U-Net per fold with early stopping; saves the best checkpoint
# to OUT_DIR/mcdropout_f<fold>_best.pth (reuse it later without re-training).
# --num-workers 0: single-process data loading, avoids the CUDA-in-forked-worker
# crash on Kaggle ("CUDA error: initialization error"). Tiny 2D slices load fast at 0.
for f in FOLDS:
    !python train_mc.py --fold {f} --dropout-p {DROPOUT_P} --out-dir {OUT_DIR} --num-workers 0

In [ ]:
# ===== test: classic metrics (dropout OFF, deterministic) =====
# Writes mcdropout_f<fold>_scores.json (same format as the baseline).
for f in FOLDS:
    !python test_mc.py --fold {f} --dropout-p {DROPOUT_P} --out-dir {OUT_DIR}

In [ ]:
# ===== mean classic scores over folds (optional; only after all 5 folds) =====
# Reuses train_eval's aggregation on the 'mcdropout' tag (reads OUT_DIR=results).
!python train_eval.py --fold-mean mcdropout

In [ ]:
# ===== uncertainty (RAW, T=1): T passes -> maps + foreground calibration =====
# Keeps dropout ON, runs MC_SAMPLES passes per slice. --temperature 1.0 forces the
# raw (un-recalibrated) result. Produces, under OUT_DIR/uncertainty/:
#   mcdropout_f<fold>_<case>.png       per-case 5-panel (image | pred/GT | entropy | MI | variance)
#   mcdropout_f<fold>_calibration.png  reliability diagrams: foreground + per class (ECE each)
#   mcdropout_f<fold>_uncertainty.json per-case scalars + foreground/per-class ECE
#   *_maps.npy                         entropy/MI/variance volumes (--save-volumes)
for f in FOLDS:
    !python uncertainty_mc.py --fold {f} --dropout-p {DROPOUT_P} --mc-samples {MC_SAMPLES} --out-dir {OUT_DIR} --temperature 1.0 --save-volumes

In [ ]:
# ===== recalibration: fit temperature T on VAL, then re-run uncertainty with it =====
# calibrate_mc fits a single T (foreground NLL, dropout OFF) and saves
# mcdropout_f<fold>_temperature.json. uncertainty_mc then AUTO-LOADS that T and
# writes a separate '_cal' output set (raw vs calibrated sit side by side).
for f in FOLDS:
    !python calibrate_mc.py   --fold {f} --dropout-p {DROPOUT_P} --out-dir {OUT_DIR}
    !python uncertainty_mc.py --fold {f} --dropout-p {DROPOUT_P} --mc-samples {MC_SAMPLES} --out-dir {OUT_DIR} --save-volumes

## Inspect the uncertainty results
The cell below prints the JSON summary (ECE per class + foreground) and renders the calibration diagram and a few per-case uncertainty panels inline.

In [ ]:
# ===== show results inline =====
import os, glob, json
from IPython.display import Image, display

unc = os.path.join(OUT_DIR, "uncertainty")

for jp in sorted(glob.glob(os.path.join(unc, "mcdropout_f*_uncertainty.json"))):
    with open(jp) as fh:
        d = json.load(fh)
    print(os.path.basename(jp), "-> calibration:", json.dumps(d["calibration"], indent=2))

print("\ncalibration diagrams:")
for p in sorted(glob.glob(os.path.join(unc, "mcdropout_f*_calibration.png"))):
    display(Image(p))

print("\nper-case uncertainty panels (first 6):")
panels = [p for p in sorted(glob.glob(os.path.join(unc, "mcdropout_f*.png")))
          if "_calibration" not in p]
for p in panels[:6]:
    display(Image(p))